# Alice & Bob resource estimator tutorial

This Jupyter notebook is a tutorial for the **Alice & Bob resource estimator**, a tool designed to estimate the physical resources required to execute quantum algorithms on Alice & Bob hardware.

The tutorial introduces the estimator’s concepts, workflow, and usage through concrete examples, with a focus on elliptic curve cryptography (ECC).

# Table of Contents
1. [Prerequisites](#Prerequisites)
2. [What Is the Alice & Bob Resource Estimator?](#What-Is-the-Alice-&-Bob-Resource-Estimator?)
3. [Installation](#Installation)
4. [Examples](#Examples)
   1. [ECC Based on Logical Counts](#ECC-based-on-logical-counts)
   2. [Estimation from a Q# File](#Estimation-from-a-Q#-file)
   3. [Estimation from a Qualtran bloq](#Estimation-from-a-Qualtran-bloq)


## Prerequisites

The Alice & Bob resource estimator provides a Python interface, and no knowledge of Rust is required. However, readers are expected to have the following background:

- Familiarity with basic Python syntax
- A basic understanding of quantum circuits and common quantum gates
- A basic understanding of cat qubits
- Some familiarity with quantum error correction (QEC)

While **prior knowledge of elliptic curve cryptography** is helpful for interpreting the example results, it is **not required** to use the resource estimator itself.


## What Is the Alice & Bob Resource Estimator?

The [Microsoft Azure Quantum Resource Estimator](https://arxiv.org/abs/2311.05801) introduces a layered framework that connects high-level quantum programs to physical resource estimates.

The **Alice & Bob resource estimator** builds on this framework by specializing the hardware and error-correction layers for Alice & Bob’s **cat-qubit architecture**.
It takes into account routing, quantum error correction, and magic-state factory implementations.
See Readme.md for more detail.

## Installation

See Readme.md


## Usage

The python package is named `anb_estimator`. Provided python has been run via 
``` dash
pixi run python script-name
```
it can be imported as usual from within any python script from this repository as




In [ ]:
# Import the Rust functionality as a Python module

import anb_estimator

The API exposes three main functions:
- `estimate_qsharp_file(...)`
- `estimate_from_qualtran(...)`
- `estimate_logical_counts(...)`
  
The first 3 arguments of these functions are (N_L, cx, ccx), the algorithmic resource count, see Readme.md.

The additional arguments are
- `frontier` — If `false`, return a unique solution as an Estimate dataclass (containing the outputs described in Reame.md). If `true`, return the Pareto frontier as a list of Estimate objects.
- `error_total` — Overall error target; mutually exclusive with `error_budget`.
- `error_budget` — Tuple `(Proba of >= 1 logical error, Probability of >= 1 faulty magic state distillation, proba of >= 1 failed rotation synthesis)` for an explicit split; mutually exclusive with "error_total".


# Examples

For the following examples to run properly, it is advised to have opened this notebook via
``` dash
pixi run jupyter notebook
```
(see Readme.md)

## ECC based on logical counts

To obtain the logical resource counts, we reproduce the calculations described in [arXiv:2302.06639](https://arxiv.org/abs/2302.06639).
Physical estimate is then obtained from it.

In [ ]:
import math

bit_size = 256  # number of qubits allocated to the two registers containing the factors of the elliptic curve multiplication
window_size = 18

# Qubit gate counts, arXiv:2302.06639 (app C.11)
qubit_count = 9 * bit_size + window_size + 4

# Asymptotic gate counts, arXiv:2302.06639 (app C.10)
cx_count = math.ceil(448 * bit_size**3 / window_size)
ccx_count = math.ceil(348 * bit_size**3 / window_size)

logical_count = anb_estimator.LogicalCounts(qubit_count, cx_count, ccx_count)

elliptic_curve = anb_estimator.estimate_logical_counts(  # type: ignore
    logical_count,
    frontier=False,
    error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0),
)

In [ ]:
print("=== Logical_estimates example ===")
print(elliptic_curve)

Note that this cell runs the program with exactly the same parameters as the Rust CLI comment
``` bash
cargo run --example=elliptic_log
```

You can access one field individually:

In [ ]:
code_distance = elliptic_curve.estimates.code_distance
print(f"Code distance: {code_distance}")
# Same for other parameters

## Estimation from a Q# file

The last option is to specify an algorithm in a Q# file and pass it as an input to the resource estimator.
To perform the optimization we simply have to provide the path to the Q# file we are interested in. An example is given in ``qsharp/Adder.rs``.

In [ ]:
filename = "../qsharp/Adder.qs"
result = anb_estimator.estimate_qsharp_file(  # type: ignore
    filename, frontier=True, error_total=None, error_budget=(0.0005, 0.0005, 0.0)
)
single_qsharp = result.estimates
frontier = result.frontier
counts = result.counts

print("=== Q# example ===")
print(single_qsharp)

Note that this cell runs the program with exactly the same parameters as the Rust CLI comment
``` bash
cargo run --example=from_qsharp
```

## Estimation from a Qualtran bloq

Another option is to estimate the resources of a [Qualtran](https://qualtran.readthedocs.io) "bloq"  by using the function ``estimate_from_qualtran`` as shown below. Qualtran produces an estimate of the logical resource counts, which is then converted to physical data by the resource estimator.

The cool thing about Qualtran is that with this interface we can easily perform a resource estimation for the large number of algorithms that are contained in the Qualtran library.
Here is another example for the QFT on a register of 100 qubits.

**Note that there is currently no direct way to use qualtran code as input to the program without using the Python API (i.e. no CLI method at the moment).**

Ex 1: Quantum Fourier Transform

In [ ]:
from qualtran.bloqs.qft import QFTTextBook

qft_text_book = QFTTextBook(100)


result_qft = anb_estimator.estimate_from_qualtran(
    qft_text_book, frontier=False, error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0)
)

print(result_qft)

Ex 2: RSA Factorisation via Shor's algorithm, see [Factoring RSA — Qualtran documentation](https://qualtran.readthedocs.io/en/latest/bloqs/cryptography/rsa/rsa.html#)


In [ ]:
from dataclasses import dataclass

from qualtran.bloqs.cryptography.rsa import ModExp
from qualtran.resource_counting import get_cost_value, QECGatesCost
from qualtran.drawing import show_bloq


# Modular exponentiation is the main computational primitive for quantum factoring algorithms
# see [How to factor 2048 bit RSA integers in 8 hours using 20 million noisy qubits](https://arxiv.org/abs/1905.09749).

# Create a ModExp bloq for a small test case
mod_exp = ModExp(base=7, mod=15, exp_bitsize=4, x_bitsize=8)


In [ ]:
result_rsa = anb_estimator.estimate_from_qualtran(
    mod_exp, frontier=False, error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0)
)

print(result_rsa)